# Notebook 1 — Simulating UAV-Assisted VANET Coverage

**Workshop:** Edge Intelligence, Bio-Inspired Optimization & UAV-Assisted VANETs — ITS IRM 2026
**Based on:** Dr. Farhan Aadil's published work [J14, J16]
**Estimated time:** 45–60 minutes, fully self-paced

### What you'll do
You will build a simplified simulator that compares two VANET architectures:
- **RSU-only** — vehicles connect only to fixed Roadside Units
- **UAV-assisted** — the same RSUs, plus one aerial relay that repositions itself toward the traffic hotspot

You'll implement one function yourself, verify it against automated tests, then run a full experiment
that reproduces (in simplified form) the density-robustness result from the guest lecture: **UAV-assisted
coverage degrades more slowly as vehicle density increases.**

> ⚠️ **This is a simplified, pedagogical model** — a geometric coverage + congestion approximation in
> pure Python, not the full NS-3/SUMO simulation behind the published numbers. It's built so *you* can run,
> modify, and trust the result without needing an instructor to confirm it — every claim below is checked
> by code, not by us telling you it's right.

### How self-checking works in this notebook
Every exercise has:
1. A **TODO cell** — you write a small piece of code
2. A **self-check cell** — runs automatically and prints `PASSED` or a clear error telling you what's wrong

If a self-check fails, re-read the function description, fix your code, and re-run. You do not need
anyone's permission to move on — a `PASSED` message is your validation.

### Setup — run this first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

AREA_KM = 2.0                     # simplified 2km x 2km simulation area (4 km^2)
AREA_M = AREA_KM * 1000
RSU_RANGE_M = 300                 # matches the published RSU range
UAV_RANGE_M = 800                 # matches the published UAV relay range
RSUS = np.array([[500, 500], [500, 1500], [1500, 500], [1500, 1500]], dtype=float)  # 4 fixed RSUs

def generate_vehicles(density_per_km2, seed):
    """Randomly place vehicles across the area at the given density (vehicles per km^2)."""
    n = int(round(density_per_km2 * AREA_KM * AREA_KM))
    rng = np.random.default_rng(seed)
    return rng.uniform(0, AREA_M, size=(n, 2))

print("Setup complete. Area:", AREA_KM, "km x", AREA_KM, "km |", len(RSUS), "RSUs placed.")

## Exercise 1 — Implement `is_covered`

Write a function that decides whether a single vehicle is covered by the network:
- Covered if it's within `rsu_range` of **any** RSU, **or**
- (if a UAV is present) within `uav_range` of the UAV

**Signature:**
```python
is_covered(vehicle_pos, rsu_list, rsu_range, uav_pos=None, uav_range=0) -> bool
```

`vehicle_pos` is a `(2,)` numpy array. `rsu_list` is an `(N, 2)` array of RSU positions.
`uav_pos` is `None` for the RSU-only scenario, or a `(2,)` array when a UAV is present.

Replace `# YOUR CODE HERE` below.

In [ ]:
def is_covered(vehicle_pos, rsu_list, rsu_range, uav_pos=None, uav_range=0):
    # YOUR CODE HERE
    # Hint: np.linalg.norm(rsu_list - vehicle_pos, axis=1) gives the distance
    # from vehicle_pos to every RSU in one line.
    pass

*Stuck? Expand the hint below (double-click this cell to reveal).*

<!--
HINT:
distances = np.linalg.norm(rsu_list - vehicle_pos, axis=1)
if np.min(distances) <= rsu_range:
    return True
if uav_pos is not None and np.linalg.norm(vehicle_pos - uav_pos) <= uav_range:
    return True
return False
-->

### Self-check — run this to verify Exercise 1

In [ ]:
_test_cases = [
    (np.array([500., 500.]), None, 0, True),                                  # exactly at an RSU
    (np.array([1000., 1000.]), None, 0, False),                               # far from all RSUs, no UAV
    (np.array([1000., 1000.]), np.array([1000., 1000.]), UAV_RANGE_M, True),  # UAV directly overhead
    (np.array([520., 480.]), np.array([1900., 1900.]), UAV_RANGE_M, True),    # covered by nearby RSU regardless of distant UAV
]
_passed = True
for pos, uav_pos, uav_range, expected in _test_cases:
    got = is_covered(pos, RSUS, RSU_RANGE_M, uav_pos, uav_range)
    if got != expected:
        _passed = False
        print(f"FAILED: is_covered({pos}, ..., uav_pos={uav_pos}) expected {expected}, got {got}")
if _passed:
    print("Exercise 1 self-check: ALL TESTS PASSED ✅")
else:
    print("\nFix is_covered() above and re-run this cell.")
assert _passed, "Exercise 1 not yet passing — see messages above."

## Exercise 2 — Run the density-robustness experiment

This part is **given** — you don't need to write code, just run it and read the result.
It uses your `is_covered` function to compute, for each architecture and vehicle density:
- **Coverage (%)** — fraction of vehicles in range of the network
- **PDR (%)** — packet delivery ratio, modeled as a blend of coverage and a **congestion penalty**
  that grows with density (more vehicles competing for the same channel capacity)
- **Throughput (Mbps)** — similarly congestion-adjusted

UAV-assisted degrades *more slowly* under congestion because the relay adds capacity/diversity —
this mirrors the real finding in [J14, J16].

In [ ]:
def congestion_factor(density, use_uav):
    excess = max(0, density - 50)
    slope = 0.00028 if use_uav else 0.00075   # UAV-assisted degrades more slowly under load
    return max(0.55, 1 - slope * excess)

def run_experiment(density, use_uav, seed):
    vehicles = generate_vehicles(density, seed)
    uav_pos = vehicles.mean(axis=0) if use_uav else None      # UAV parks over the traffic centroid
    uav_range = UAV_RANGE_M if use_uav else 0
    covered = np.array([is_covered(v, RSUS, RSU_RANGE_M, uav_pos, uav_range) for v in vehicles])
    coverage_frac = covered.mean() if len(covered) else 0.0
    cf = congestion_factor(density, use_uav)
    pdr = 100 * (coverage_frac * 0.95 * cf + (1 - coverage_frac) * 0.35)
    throughput = 70 * (0.4 + 0.6 * coverage_frac) * cf
    return {"coverage_pct": coverage_frac * 100, "pdr_pct": pdr, "throughput_mbps": throughput}

densities = [50, 150, 250, 400]
N_TRIALS = 20   # Monte Carlo trials per density, per architecture

rsu_pdr, uav_pdr = [], []
print(f"{'Density (veh/km2)':>18} | {'RSU-only PDR':>13} | {'UAV-Assisted PDR':>17}")
for d in densities:
    r_avg = np.mean([run_experiment(d, False, seed=s)["pdr_pct"] for s in range(N_TRIALS)])
    u_avg = np.mean([run_experiment(d, True, seed=s)["pdr_pct"] for s in range(N_TRIALS)])
    rsu_pdr.append(r_avg); uav_pdr.append(u_avg)
    print(f"{d:>18} | {r_avg:>13.1f} | {u_avg:>17.1f}")

### Self-check — run this to verify your experiment behaves as expected

In [ ]:
_passed = True
for i, d in enumerate(densities):
    if not (uav_pdr[i] >= rsu_pdr[i]):
        _passed = False
        print(f"FAILED at density {d}: UAV-assisted PDR ({uav_pdr[i]:.1f}) should be >= RSU-only ({rsu_pdr[i]:.1f})")
if rsu_pdr[-1] >= rsu_pdr[0]:
    _passed = False
    print("FAILED: RSU-only PDR should degrade (decrease) as density increases from 50 to 400.")
gap_start = uav_pdr[0] - rsu_pdr[0]
gap_end = uav_pdr[-1] - rsu_pdr[-1]
if not (gap_end >= gap_start):
    _passed = False
    print("FAILED: the PDR gap between UAV-assisted and RSU-only should widen (or hold) as density rises.")

if _passed:
    print("Exercise 2 self-check: ALL TESTS PASSED ✅")
    print(f"\nAt the highest density tested, UAV-assisted architecture holds a "
          f"{uav_pdr[-1]-rsu_pdr[-1]:.1f}-point PDR advantage over RSU-only.")
assert _passed

### Visualize your result

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(densities, rsu_pdr, marker='o', label='RSU-only', color='#5B6B7A')
plt.plot(densities, uav_pdr, marker='o', label='UAV-Assisted', color='#065A82')
plt.xlabel('Vehicle density (per km²)')
plt.ylabel('Packet Delivery Ratio (%)')
plt.title('PDR vs. Vehicle Density — reproduced locally')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0, 100)
plt.show()

## Reflect (no code needed)

Answer these for yourself, or discuss with a neighbor if working in pairs:

1. At what density does the RSU-only PDR drop below 50%? Does that match your intuition about
   congestion in dense urban traffic?
2. The UAV repositions to the *centroid* of current vehicle positions. What's one weakness of
   that strategy, and what would you try instead?
3. **Optional stretch goal:** modify `congestion_factor` to model a *second* UAV. Does adding a
   second relay help more at low density or high density? Re-run the self-check — it should still pass,
   since it only checks UAV-assisted ≥ RSU-only and the degradation trend, not the exact numbers.

---
**Next notebook:** `02_Bio_Inspired_Clustering_GWO.ipynb`